# Patched Simulations

## Imports

In [8]:
# Important modules
import os,sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches

sys.path.append('../../tools/')

os.environ['JAXIONS_DIR'] = '/Users/Mathieu/Salsa/jaxionsdir/'
sys.path.append('/Users/Mathieu/Salsa/jaxionsdir/jaxions/scripts/') #adapt to your JAXIONS_DIR

# Kinetic misalignment functions
from kin_mis_utils import evolution_kinetic_mis

# Deal with the measurement files from the simulations
from pyaxions import jaxions as pa
from pyaxions import spaxcreate as sp
from pyaxions import spectrum as spec
import importlib
importlib.reload(pa)

# Some plot settings
import matplotlib as mpl
mpl.rcParams['mathtext.rm'] = 'serif'
mpl.rcParams['mathtext.fontset'] = 'cm'
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['font.size'] = 16
mpl.rcParams['text.usetex'] = True

## Parameters

In [5]:
# Simulation parameters
N = 1024 # Points/dimension
L_list = [2, 4] # Box lengths in L₁ units

# Axion parameters
fAGeV = 1e+13 # Axion decay constant in GeV
theta1 = 2.37 # Initial field value of zero mode
vheta1 = 100 # Initial velocity value of zero mode
n = 8 # QCD approximation

tautab = np.linspace(1, 5, 1000) # Normalized conformal time array, SHOULD START AT 1!

## Characteristic Length Scale

In [9]:
temp = evolution_kinetic_mis(fAGeV, theta1, vheta1, tautab)

In [10]:
# Axion mass becoming cosmologically relevant
H1MeV = temp['H1MeV']
R1 = temp['R1']

In [11]:
# Trapping
TMeV_trap = temp['TMeV_trap']
HMeV_trap = temp['HMeV_trap']
R_trap = temp['R_trap']
mAMeV_trap = temp['mAMeV_trap']

In [12]:
L1 = 1/(H1MeV*R1) # L₁

k_star = H1MeV * (vheta1**2/2)**((n/2+1)/(n+6)) # Characteristic comoving momentum scale

L_factor = 2*np.pi*10 / (k_star * R1) / L1 # Scaled length in L₁ units

print(L_factor)

2.9999854785807356


## Patching

In [14]:
# Load measurement file
path = os.getcwd()

file_path = f"/out_N{N}_L{L_list[0]}_fA{fAGeV:.0e}_theta{theta1:.2f}_vheta{vheta1:.2f}"
mf = pa.findmfiles(path +  file_path)
mask = pa.gml(mf, 'psp?')

# Measurements
R = pa.gml(mf, 'R') # Scale factor in R₁
tau = pa.gml(mf, 'ct') # Conformal time 
T = pa.gml(mf, 'T') # Temperature in MeV
theta_maps = pa.gml(mf, 'maptheta') # Field 
vheta_maps = (pa.gml(mf, 'mapvheta')-theta_maps)/R[:, None, None] # Velocity w.r.t. conformal time

[]


In [ ]:
spectra_dict = {}

for L in L_list:
    file_path = f"/out_N{N}_L{L}_fA{fAGeV:.0e}_theta{theta1:.2f}_vheta{vheta1:.2f}"
    mf = pa.findmfiles(path +  file_path)
    mask = pa.gml(mf, 'psp?')
    
    tau = pa.gml(mf, 'ct') # Conformal time
    eA = pa.gml(mf, 'eA') # Axion energy
    sizeN = pa.gm(mf[0], 'N') # Points/dimension
    nmodes = pa.phasespacedensityBOX(sizeN) # Phase space modes (averages over # of modes in each shell)
    psps = pa.gml(mf[mask], 'psp')/(eA[mask, None]**2 * nmodes) # Axion energy density contrast spectrum
    s = spec.espevol(mf) # Spectral values
    k_values = s.avek # Momentum values

    # New units 
    kappa_star = k_values * R1*H1MeV / (R_trap*maMeV_trap)
    tau_star = 2*HMeV_trap * (((tau**2)/2)/H1MeV)

    spectra_dict[L] = {
        'tau': tau[mask],
        'k': k_values,
        'tau_star': tau_star[mask],
        'kappa_star': kappa_star,
        'psps': psps
    }

In [ ]:
# Choose time points
desired_tau_values = [1.0, 2.0, 3.0, 5.0]

In [ ]:
fig, ax = plt.subplots(figsize = (10, 6))

# Color map for times
colors = plt.cm.viridis(np.linspace(0, 1, len(desired_tau_values)))

# Define linestyles for each L
linestyles = ['-', '--', '-.', ':']
linestyle_map = {L: ls for L, ls in zip(spectra_dict.keys(), linestyles)}

# Legend handles
time_handles = []
L_handles = []

for time_idx, desired_tau in enumerate(desired_tau_values):
    color = colors[time_idx]

    for L, data in spectra_dict.items():
        tau = data['tau']
        k = data['k']
        tau_star = data['tau_star']
        kappa_star = data['kappa_star']
        psps = data['psps']
        
        linestyle = linestyle_map[L]

        if len(tau) == 0:
            continue

        idx = np.argmin(np.abs(tau - desired_tau))
        if idx >= psps.shape[0]:
            continue

        spectrum = k**3 / np.pi**2 * psps[idx]
        
        ax.loglog(kappa_star, spectrum, color = color, linestyle = linestyle)

# Build legend for time (colors)
for time_idx, desired_tau in enumerate(desired_tau_values):
    color = colors[time_idx]
    patch = mpatches.Patch(color = color, label = fr"$\tau={desired_tau:.2f}$")
    time_handles.append(patch)

# Build legend for L (linestyles)
for L, linestyle in linestyle_map.items():
    line = mlines.Line2D([], [], color = 'black', linestyle = linestyle, label = fr"$L={L}$")
    L_handles.append(line)

# Add legends
first_legend = ax.legend(handles = time_handles, loc = 'upper right')
second_legend = ax.legend(handles = L_handles, loc = 'lower left')
ax.add_artist(first_legend) # Add the first legend manually

ax.set_title(
    r'Dimensionless Power Spectrum of Density Fluctuations' + '\n' + 
    r'$f_a = {:.0e}$ GeV, $\Theta_1 = {:.2f}$, $\dot{{\Theta}}_1/H_1 = {:.2f}$'.format(fAGeV, theta1, vheta1),
    fontsize = 14,
    fontweight = 'bold'
)
ax.set_xlabel(r'$\kappa_\ast$')
ax.set_ylabel(r'$\Delta_{\kappa_{\ast}}^2$')
ax.set_yscale('log')
ax.grid(True, linestyle = ':', alpha = 0.5)

# Add secondary x-axis
secax = ax.secondary_xaxis('top', functions = (lambda x: np.interp(x, kappa_star, k_values), lambda x: np.interp(x, k_values, kappa_star)))
secax.set_xlabel(r'$k$')

# Avoid overlapping ticks
ticks = secax.get_xticks()
min_spacing = 10**-10 * (ticks[-1] - ticks[0]) # Define minimum spacing threshold
# Filter out ticks that are closer than min_spacing to the previous tick
filtered_ticks = [ticks[0]]
for t in ticks[1:]:
    if t - filtered_ticks[-1] > min_spacing:
        filtered_ticks.append(t)
secax.set_xticks(filtered_ticks)

plt.tight_layout()
plt.show()
plt.close()